In [83]:
import numpy as np
import pandas as pd

In [84]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [85]:
House= pd.read_csv("House Price Prediction Dataset.csv")

In [86]:
House.head()

,ID,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


In [87]:
House.index

RangeIndex(start=0, stop=2000, step=1)

In [88]:
House.columns

Index(['ID', 'Area', 'Bedrooms', 'Bathrooms', 'Floors', 'YearBuilt',
       'Location', 'Condition', 'Garage', 'Price'],
      dtype='object')

In [89]:
House.iloc[4]

ID                  5
Area             4926
Bedrooms            1
Bathrooms           4
Floors              2
YearBuilt        1975
Location     Downtown
Condition        Fair
Garage            Yes
Price          636056
Name: 4, dtype: object

In [90]:
House.iloc[:,4]

0       3
1       3
2       3
3       2
4       2
       ..
1995    3
1996    1
1997    2
1998    2
1999    3
Name: Floors, Length: 2000, dtype: int64

In [91]:
House.iloc[:,0]

0          1
1          2
2          3
3          4
4          5
        ... 
1995    1996
1996    1997
1997    1998
1998    1999
1999    2000
Name: ID, Length: 2000, dtype: int64

In [92]:
House.drop(columns=['ID'],inplace=True)

In [93]:
House.head()

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3592,2,2,3,1938,Downtown,Good,No,266746
3,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,4926,1,4,2,1975,Downtown,Fair,Yes,636056


# Step 1 of pipeline i want to drop the data

# Id is dropped 

In [94]:
House.head()

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3592,2,2,3,1938,Downtown,Good,No,266746
3,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,4926,1,4,2,1975,Downtown,Fair,Yes,636056


1. Train test split kiye uske baad 
2. 

In [95]:
X_train,x_test,Y_train,y_test = train_test_split(
    House.drop(columns=['Price']), # Removed price prediction form training 
    House['Price'], # target Y for output 
    test_size=0.4, # 40 % data
    random_state=52    
    )

In [96]:
X_train.head()

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage
1892,3219,5,3,1,1979,Rural,Fair,No
1997,1062,5,1,2,1903,Rural,Poor,No
43,1863,3,3,1,2006,Urban,Excellent,No
1847,905,1,3,1,1942,Urban,Excellent,No
1525,513,4,3,3,1948,Rural,Fair,Yes


In [97]:
x_test.head(5)

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage
8,630,2,2,1,1932,Rural,Poor,Yes
1932,2793,4,3,2,1980,Suburban,Good,No
1937,741,4,4,3,1951,Rural,Good,Yes
1401,3996,3,4,3,1961,Downtown,Fair,Yes
979,3952,2,1,1,1977,Rural,Fair,Yes


In [98]:
Y_train.head(5)

1892    370098
1997    476925
43      643167
1847    687096
1525    927709
Name: Price, dtype: int64

In [99]:
y_test.head(5)

8       652878
1932    350646
1937    663337
1401    526728
979     741419
Name: Price, dtype: int64

#### Note that acutal data se 2 step agge chal raha hai 

## Now Transfromers
    1. no missing data  

# min max scaller 
    1. Aera 

YOu can Also use scale to scale down your value between a range 

In [100]:
trf_Aera = ColumnTransformer([
    ('area_scaler', MinMaxScaler(feature_range=(0,50)), [1])  # [1] is the column index for 'Area'
], remainder='passthrough')

# same min max for 
    * Bathrom, BEDSroom , Floors 
    * slice(1,4)
    

In [101]:
trf_b_b_f = ColumnTransformer([
    ('Bed_Bath_Floor_scaler', MinMaxScaler(feature_range=(0,5)), slice(1,4))  # [1] is the column index for 'Area'
], remainder='passthrough')

In [102]:
print(House.columns[1:4])

Index(['Bedrooms', 'Bathrooms', 'Floors'], dtype='object')


# min max scaling for 
Year buit

In [103]:
trf_year=ColumnTransformer([
   ('year_built', MinMaxScaler(feature_range=(0, 124)),[5])
],remainder= "passthrough")


# one hot encoding 
Location	Condition	Garage


# OneHotEncoder creates:

    * Rural: [1,0,0,0]
    * Downtown: [0,1,0,0]
    * Suburban: [0,0,1,0]
    * Urban: [0,0,0,1]

In [113]:
trf_ohe = ColumnTransformer(
    [
    ('location',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[5]),
    ('condition',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[6]),
    ('garage',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[7]),
    ],remainder="passthrough"
)

In [114]:
print(House.columns[5:8])

Index(['Location', 'Condition', 'Garage'], dtype='object')


In [115]:
trf4 = SelectKBest(score_func=chi2,k=8)

In [116]:
trf5 = DecisionTreeClassifier()

In [ ]:
Pipe_house = 